In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p7-1/adaptive_dispatch_evaluation.csv
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p7-1/adaptive_dispatch_threshold.csv
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p7-1/adaptive_hourly_dispatch_schedule.csv
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/calibrated_pressure_predictions.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/events_with_spatial_temporal_candidates.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/zone_lookup_250m.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/directed_candidate_graph_1000m.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/zone_time_model_table.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/zone_observation_confidence.csv
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/local_pressure_explanations.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-ro

# Phase 7

## Forecast Feature Ablations


In [2]:
from pathlib import Path
import gc
import json

import pyarrow.parquet as pq
import torch

from catboost import CatBoostRegressor

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

OUTPUT_DIR = Path("/kaggle/working/parking_phase7")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_input_file(filename):
    matches = list(Path("/kaggle/input").rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found."
        )

    return matches[0]


MODEL_TABLE_PATH = find_input_file(
    "zone_time_model_table.parquet"
)

schema_columns = pq.ParquetFile(
    MODEL_TABLE_PATH
).schema.names

explicit_features = [
    "zone_index",
    "incident_count",
    "unique_vehicles",
    "named_junction_events",
    "main_road_events",
    "road_crossing_events",
    "footpath_events",
    "large_vehicle_events",
    "multi_offence_events",
    "main_road_share",
    "road_crossing_share",
    "footpath_share",
    "named_junction_share",
    "large_vehicle_share",
    "multi_offence_share",
    "history_hours_available",
    "rolling_std_24h",
    "same_hour_7d_mean",
    "same_hour_7d_active_rate",
    "same_weekhour_4w_mean",
    "consecutive_active_hours_before",
    "short_term_trend",
    "current_burst_z",
    "relative_to_same_hour"
]

dynamic_prefixes = (
    "incident_lag_",
    "unique_vehicles_lag_",
    "past_incidents_",
    "rolling_mean_",
    "active_fraction_"
)

dynamic_features = [
    col for col in schema_columns
    if col.startswith(dynamic_prefixes)
]

feature_columns = list(dict.fromkeys([
    col
    for col in explicit_features + dynamic_features
    if col in schema_columns
]))

load_columns = list(dict.fromkeys(
    feature_columns
    + [
        "time_window",
        "target_time",
        "split_code",
        "target_next_hour_pressure"
    ]
))

model_data = pd.read_parquet(
    MODEL_TABLE_PATH,
    columns=load_columns,
    filters=[
        ("split_code", "in", [1, 2, 3])
    ]
)

model_data["hour_of_day"] = (
    model_data["time_window"]
    .dt.hour
    .astype("int8")
)

model_data["day_of_week"] = (
    model_data["time_window"]
    .dt.dayofweek
    .astype("int8")
)

model_data["month"] = (
    model_data["time_window"]
    .dt.month
    .astype("int8")
)

model_data["is_weekend"] = (
    model_data["day_of_week"] >= 5
).astype("int8")

feature_columns += [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend"
]

base_categorical_features = [
    "zone_index",
    "hour_of_day",
    "day_of_week",
    "month"
]

print("Rows:", len(model_data))
print("Full features:", len(feature_columns))
print(
    "Memory MB:",
    model_data.memory_usage(deep=True).sum()
    / 1024**2
)

Rows: 4018165
Full features: 49
Memory MB: 770.2362985610962


In [3]:
# Define feature groups and a common training sample

# Every model uses the exact same positive observations, sampled zero observations and sample weights.
# This ensures differences are caused by feature removal rather than different training data

RANDOM_STATE = 42
NEGATIVE_TO_POSITIVE_RATIO = 12

recurrence_features = [
    col for col in feature_columns
    if (
        col.startswith(dynamic_prefixes)
        or col in [
            "rolling_std_24h",
            "same_hour_7d_mean",
            "same_hour_7d_active_rate",
            "same_weekhour_4w_mean",
            "consecutive_active_hours_before",
            "short_term_trend",
            "current_burst_z",
            "relative_to_same_hour"
        ]
    )
]

vehicle_features = [
    col for col in feature_columns
    if (
        col == "unique_vehicles"
        or col.startswith("unique_vehicles_lag_")
        or col in [
            "large_vehicle_events",
            "large_vehicle_share"
        ]
    )
]

context_features = [
    col for col in feature_columns
    if col in [
        "main_road_events",
        "road_crossing_events",
        "footpath_events",
        "named_junction_events",
        "multi_offence_events",
        "main_road_share",
        "road_crossing_share",
        "footpath_share",
        "named_junction_share",
        "multi_offence_share"
    ]
]

heatmap_like_features = [
    col for col in [
        "zone_index",
        "hour_of_day",
        "day_of_week",
        "month",
        "is_weekend",
        "rolling_mean_168h",
        "past_incidents_168h",
        "active_fraction_168h",
        "same_hour_7d_mean",
        "same_hour_7d_active_rate",
        "same_weekhour_4w_mean"
    ]
    if col in feature_columns
]

feature_sets = {
    "full_model": feature_columns,

    "static_heatmap_like":
        heatmap_like_features,

    "no_temporal_recurrence": [
        col for col in feature_columns
        if col not in recurrence_features
    ],

    "no_vehicle_information": [
        col for col in feature_columns
        if col not in vehicle_features
    ],

    "no_context_semantics": [
        col for col in feature_columns
        if col not in context_features
    ],

    "no_zone_identity": [
        col for col in feature_columns
        if col != "zone_index"
    ]
}

train_all = model_data.loc[
    model_data["split_code"] == 1
]

positive_index = train_all.index[
    train_all[
        "target_next_hour_pressure"
    ] > 0
].to_numpy()

zero_index = train_all.index[
    train_all[
        "target_next_hour_pressure"
    ] == 0
].to_numpy()

rng = np.random.default_rng(
    RANDOM_STATE
)

n_sampled_zeros = min(
    len(zero_index),
    NEGATIVE_TO_POSITIVE_RATIO
    * len(positive_index)
)

sampled_zero_index = rng.choice(
    zero_index,
    size=n_sampled_zeros,
    replace=False
)

training_index = np.concatenate([
    positive_index,
    sampled_zero_index
])

rng.shuffle(training_index)

train_sample = model_data.loc[
    training_index
].copy()

y_train = train_sample[
    "target_next_hour_pressure"
].to_numpy(dtype=np.float32)

zero_sampling_correction = (
    len(zero_index)
    / n_sampled_zeros
)

sampling_weight = np.where(
    y_train == 0,
    zero_sampling_correction,
    1.0
)

pressure_weight = (
    1 + np.sqrt(y_train)
)

train_weights = (
    sampling_weight
    * pressure_weight
).astype("float32")

y_train_log = np.log1p(
    y_train
).astype("float32")

print("Training sample:", len(train_sample))
print("Positive rows:", len(positive_index))
print("Sampled zero rows:", n_sampled_zeros)

display(
    pd.DataFrame({
        "ablation": feature_sets.keys(),
        "feature_count": [
            len(features)
            for features in feature_sets.values()
        ]
    })
)

Training sample: 690547
Positive rows: 53119
Sampled zero rows: 637428


,ablation,feature_count
0,full_model,49
1,static_heatmap_like,11
2,no_temporal_recurrence,20
3,no_vehicle_information,43
4,no_context_semantics,39
5,no_zone_identity,48


In [4]:
# common evaluation function
HOTSPOT_THRESHOLD = 8
TOP_K = 25


def prepare_features(
    data,
    selected_features
):
    X = data[
        selected_features
    ].copy()

    categorical = [
        col
        for col in base_categorical_features
        if col in selected_features
    ]

    for col in categorical:
        X[col] = X[col].astype(str)

    return X, categorical


def evaluate_prediction(
    target,
    prediction,
    model_name,
    split_name
):
    target_flat = target.ravel()
    prediction_flat = prediction.ravel()

    active = target_flat > 0

    top_indices = np.argpartition(
        -prediction,
        kth=TOP_K - 1,
        axis=1
    )[:, :TOP_K]

    selected_prediction = (
        np.take_along_axis(
            prediction,
            top_indices,
            axis=1
        )
    )

    ordering = np.argsort(
        -selected_prediction,
        axis=1
    )

    top_indices = np.take_along_axis(
        top_indices,
        ordering,
        axis=1
    )

    selected_target = np.take_along_axis(
        target,
        top_indices,
        axis=1
    )

    ideal_target = np.sort(
        target,
        axis=1
    )[:, ::-1][:, :TOP_K]

    discounts = (
        1
        / np.log2(
            np.arange(2, TOP_K + 2)
        )
    ).astype("float32")

    dcg = (
        selected_target
        * discounts
    ).sum(axis=1)

    ideal_dcg = (
        ideal_target
        * discounts
    ).sum(axis=1)

    valid_ndcg = ideal_dcg > 0

    true_hotspots = (
        target >= HOTSPOT_THRESHOLD
    )

    selected_hotspots = (
        selected_target
        >= HOTSPOT_THRESHOLD
    )

    hotspot_hours = (
        true_hotspots.any(axis=1)
    )

    return {
        "model": model_name,
        "split": split_name,
        "feature_count": None,

        "overall_mae": np.mean(
            np.abs(
                target_flat
                - prediction_flat
            )
        ),

        "active_target_mae": np.mean(
            np.abs(
                target_flat[active]
                - prediction_flat[active]
            )
        ),

        "pressure_capture_at_25": (
            selected_target.sum()
            / target.sum()
        ),

        "active_zone_recall_at_25": (
            (selected_target > 0).sum()
            / (target > 0).sum()
        ),

        "hotspot_recall_at_25": (
            selected_hotspots.sum()
            / true_hotspots.sum()
            if true_hotspots.sum() > 0
            else 0
        ),

        "hotspot_hit_rate_at_25": (
            selected_hotspots[
                hotspot_hours
            ].any(axis=1).mean()
            if hotspot_hours.any()
            else 0
        ),

        "mean_ndcg_at_25": (
            (
                dcg[valid_ndcg]
                / ideal_dcg[valid_ndcg]
            ).mean()
            if valid_ndcg.any()
            else 0
        )
    }

In [5]:
# train and evaluate all ablation models
use_gpu = torch.cuda.is_available()

ablation_rows = []
importance_rows = []

for model_name, selected_features in (
    feature_sets.items()
):
    print(
        f"\nTraining {model_name} "
        f"with {len(selected_features)} features"
    )

    X_train, categorical_features = (
        prepare_features(
            train_sample,
            selected_features
        )
    )

    model_parameters = {
        "iterations": 150,
        "depth": 8,
        "learning_rate": 0.05,
        "loss_function": "RMSE",
        "l2_leaf_reg": 5,
        "random_seed": RANDOM_STATE,
        "allow_writing_files": False,
        "verbose": False
    }

    if use_gpu:
        model_parameters.update({
            "task_type": "GPU",
            "devices": "0"
        })

    ablation_model = CatBoostRegressor(
        **model_parameters
    )

    ablation_model.fit(
        X_train,
        y_train_log,
        sample_weight=train_weights,
        cat_features=categorical_features
    )

    importances = (
        ablation_model
        .get_feature_importance()
    )

    for feature, importance in zip(
        selected_features,
        importances
    ):
        importance_rows.append({
            "model": model_name,
            "feature": feature,
            "importance": importance
        })

    del X_train
    gc.collect()

    for split_code, split_name in [
        (2, "validation"),
        (3, "test")
    ]:
        split_data = (
            model_data.loc[
                model_data["split_code"]
                == split_code
            ]
            .sort_values([
                "target_time",
                "zone_index"
            ])
        )

        X_split, _ = prepare_features(
            split_data,
            selected_features
        )

        predicted_pressure = np.clip(
            np.expm1(
                ablation_model.predict(
                    X_split
                )
            ),
            0,
            None
        ).astype("float32")

        n_hours = split_data[
            "target_time"
        ].nunique()

        n_zones = split_data[
            "zone_index"
        ].nunique()

        target_matrix = (
            split_data[
                "target_next_hour_pressure"
            ]
            .to_numpy(dtype=np.float32)
            .reshape(
                n_hours,
                n_zones
            )
        )

        prediction_matrix = (
            predicted_pressure.reshape(
                n_hours,
                n_zones
            )
        )

        result = evaluate_prediction(
            target=target_matrix,
            prediction=prediction_matrix,
            model_name=model_name,
            split_name=split_name
        )

        result["feature_count"] = len(
            selected_features
        )

        ablation_rows.append(result)

        del X_split
        gc.collect()

    del ablation_model
    gc.collect()

ablation_results = pd.DataFrame(
    ablation_rows
)

ablation_feature_importance = (
    pd.DataFrame(importance_rows)
)

display(
    ablation_results.sort_values(
        [
            "split",
            "pressure_capture_at_25"
        ],
        ascending=[True, False]
    )
)


Training full_model with 49 features

Training static_heatmap_like with 11 features

Training no_temporal_recurrence with 20 features

Training no_vehicle_information with 43 features

Training no_context_semantics with 39 features

Training no_zone_identity with 48 features


,model,split,feature_count,overall_mae,active_target_mae,pressure_capture_at_25,active_zone_recall_at_25,hotspot_recall_at_25,hotspot_hit_rate_at_25,mean_ndcg_at_25
7,no_vehicle_information,test,43,0.1029,2.9213,0.3901,0.2531,0.4872,0.8035,0.3650
9,no_context_semantics,test,39,0.1037,2.9174,0.3901,0.2529,0.4896,0.8094,0.3667
1,full_model,test,49,0.1015,2.9229,0.3868,0.2532,0.4840,0.7947,0.3630
11,no_zone_identity,test,48,0.1042,2.9265,0.3853,0.2511,0.4800,0.8006,0.3617
5,no_temporal_recurrence,test,20,0.1055,2.9482,0.3671,0.2464,0.4545,0.7801,0.3339
3,static_heatmap_like,test,11,0.1098,2.9994,0.3497,0.2212,0.4441,0.7742,0.3082
6,no_vehicle_information,validation,43,0.1051,2.9993,0.3790,0.2441,0.4728,0.8225,0.3417
0,full_model,validation,49,0.1040,3.0015,0.3787,0.2442,0.4672,0.8225,0.3387
8,no_context_semantics,validation,39,0.1056,3.0014,0.3768,0.2428,0.4680,0.8136,0.3371
10,no_zone_identity,validation,48,0.1061,3.0075,0.3746,0.2439,0.4655,0.8166,0.3359


In [6]:
# Compare test degradation and save
# reports how much each ablation retains relative to the full model

test_ablation = (
    ablation_results.loc[
        ablation_results["split"]
        == "test"
    ]
    .copy()
)

full_test = test_ablation.loc[
    test_ablation["model"]
    == "full_model"
].iloc[0]

higher_is_better = [
    "pressure_capture_at_25",
    "active_zone_recall_at_25",
    "hotspot_recall_at_25",
    "hotspot_hit_rate_at_25",
    "mean_ndcg_at_25"
]

for metric in higher_is_better:
    test_ablation[
        f"{metric}_retained_percent"
    ] = (
        100
        * test_ablation[metric]
        / full_test[metric]
    )

test_ablation[
    "active_mae_change_vs_full"
] = (
    test_ablation[
        "active_target_mae"
    ]
    - full_test[
        "active_target_mae"
    ]
)

test_ablation_comparison = (
    test_ablation.sort_values(
        "pressure_capture_at_25",
        ascending=False
    )
)

display(
    test_ablation_comparison[
        [
            "model",
            "feature_count",
            "overall_mae",
            "active_target_mae",
            "pressure_capture_at_25",
            "pressure_capture_at_25_retained_percent",
            "hotspot_recall_at_25",
            "hotspot_recall_at_25_retained_percent",
            "hotspot_hit_rate_at_25",
            "mean_ndcg_at_25",
            "active_mae_change_vs_full"
        ]
    ]
)

ablation_results.to_csv(
    OUTPUT_DIR
    / "forecast_feature_ablation_results.csv",
    index=False
)

test_ablation_comparison.to_csv(
    OUTPUT_DIR
    / "forecast_test_ablation_comparison.csv",
    index=False
)

ablation_feature_importance.to_csv(
    OUTPUT_DIR
    / "ablation_feature_importance.csv",
    index=False
)

with open(
    OUTPUT_DIR / "ablation_feature_manifest.json",
    "w"
) as file:
    json.dump(
        feature_sets,
        file,
        indent=2
    )

print(
    "Ablation outputs saved to:",
    OUTPUT_DIR
)

,model,feature_count,overall_mae,active_target_mae,pressure_capture_at_25,pressure_capture_at_25_retained_percent,hotspot_recall_at_25,hotspot_recall_at_25_retained_percent,hotspot_hit_rate_at_25,mean_ndcg_at_25,active_mae_change_vs_full
7,no_vehicle_information,43,0.1029,2.9213,0.3901,100.8482,0.4872,100.6601,0.8035,0.3650,-0.0016
9,no_context_semantics,39,0.1037,2.9174,0.3901,100.8347,0.4896,101.1551,0.8094,0.3667,-0.0055
1,full_model,49,0.1015,2.9229,0.3868,100.0000,0.4840,100.0000,0.7947,0.3630,0.0000
11,no_zone_identity,48,0.1042,2.9265,0.3853,99.5894,0.4800,99.1749,0.8006,0.3617,0.0036
5,no_temporal_recurrence,20,0.1055,2.9482,0.3671,94.9041,0.4545,93.8944,0.7801,0.3339,0.0253
3,static_heatmap_like,11,0.1098,2.9994,0.3497,90.3871,0.4441,91.7492,0.7742,0.3082,0.0765


Ablation outputs saved to: /kaggle/working/parking_phase7


## Spatial Holdout Validation

In [7]:
# Create five geographic holdout regions

from sklearn.cluster import KMeans

ZONE_LOOKUP_PATH = find_input_file(
    "zone_lookup_250m.parquet"
)

zone_lookup = pd.read_parquet(
    ZONE_LOOKUP_PATH
)

required_spatial_columns = [
    "zone_index",
    "centroid_x_m",
    "centroid_y_m"
]

missing_spatial_columns = [
    col for col in required_spatial_columns
    if col not in zone_lookup.columns
]

if missing_spatial_columns:
    raise KeyError(
        "Missing zone lookup columns: "
        f"{missing_spatial_columns}"
    )

SPATIAL_FOLDS = 5

coordinates = zone_lookup[
    ["centroid_x_m", "centroid_y_m"]
].to_numpy(dtype=np.float64)

coordinate_mean = coordinates.mean(axis=0)
coordinate_std = coordinates.std(axis=0)

standardized_coordinates = (
    coordinates - coordinate_mean
) / coordinate_std

spatial_clusterer = KMeans(
    n_clusters=SPATIAL_FOLDS,
    random_state=42,
    n_init=20
)

zone_lookup["spatial_fold"] = (
    spatial_clusterer
    .fit_predict(
        standardized_coordinates
    )
    .astype("int8")
)

spatial_fold_map = zone_lookup[
    ["zone_index", "spatial_fold"]
].copy()

spatial_fold_summary = (
    zone_lookup.groupby("spatial_fold")
    .agg(
        zones=("zone_index", "size"),
        minimum_x=("centroid_x_m", "min"),
        maximum_x=("centroid_x_m", "max"),
        minimum_y=("centroid_y_m", "min"),
        maximum_y=("centroid_y_m", "max")
    )
    .reset_index()
)

display(spatial_fold_summary)

assert spatial_fold_summary["zones"].sum() == 1163
assert spatial_fold_summary["zones"].min() > 0

,spatial_fold,zones,minimum_x,maximum_x,minimum_y,maximum_y
0,0,219,"774,625.0000","792,875.0000","1,417,375.0000","1,432,125.0000"
1,1,285,"766,875.0000","779,875.0000","1,428,625.0000","1,446,125.0000"
2,2,201,"786,875.0000","799,625.0000","1,425,875.0000","1,444,625.0000"
3,3,78,"776,375.0000","794,375.0000","1,443,625.0000","1,466,125.0000"
4,4,380,"777,375.0000","786,875.0000","1,431,625.0000","1,443,375.0000"


In [8]:
# Define spatial evaluation metrics

# The evaluation selects a proportional number of zones from each held-out region. 
# Since 25 of 1,163 citywide zones are normally selected, a region of roughly 230 zones is evaluated at approximately top five
def evaluate_spatial_block(
    target,
    prediction,
    model_name,
    split_name,
    fold_id,
    top_k
):
    target_flat = target.ravel()
    prediction_flat = prediction.ravel()

    active_mask = target_flat > 0

    candidate_indices = np.argpartition(
        -prediction,
        kth=top_k - 1,
        axis=1
    )[:, :top_k]

    candidate_predictions = np.take_along_axis(
        prediction,
        candidate_indices,
        axis=1
    )

    ordering = np.argsort(
        -candidate_predictions,
        axis=1
    )

    top_indices = np.take_along_axis(
        candidate_indices,
        ordering,
        axis=1
    )

    selected_target = np.take_along_axis(
        target,
        top_indices,
        axis=1
    )

    ideal_target = np.sort(
        target,
        axis=1
    )[:, ::-1][:, :top_k]

    discounts = (
        1 / np.log2(
            np.arange(2, top_k + 2)
        )
    ).astype("float32")

    dcg = (
        selected_target * discounts
    ).sum(axis=1)

    ideal_dcg = (
        ideal_target * discounts
    ).sum(axis=1)

    valid_ndcg = ideal_dcg > 0

    true_hotspots = target >= 8
    selected_hotspots = selected_target >= 8

    hotspot_hours = true_hotspots.any(axis=1)

    result = {
        "spatial_fold": fold_id,
        "split": split_name,
        "variant": model_name,
        "heldout_zones": target.shape[1],
        "top_k": top_k,

        "row_count": target_flat.size,
        "active_count": int(active_mask.sum()),
        "absolute_error_sum": float(
            np.abs(
                target_flat - prediction_flat
            ).sum()
        ),
        "active_absolute_error_sum": float(
            np.abs(
                target_flat[active_mask]
                - prediction_flat[active_mask]
            ).sum()
        ),

        "total_pressure": float(target.sum()),
        "selected_pressure": float(
            selected_target.sum()
        ),

        "selected_active": int(
            (selected_target > 0).sum()
        ),

        "total_hotspots": int(
            true_hotspots.sum()
        ),
        "selected_hotspots": int(
            selected_hotspots.sum()
        ),

        "hotspot_hours": int(
            hotspot_hours.sum()
        ),
        "hit_hotspot_hours": int(
            selected_hotspots[
                hotspot_hours
            ].any(axis=1).sum()
        ),

        "valid_ndcg_hours": int(
            valid_ndcg.sum()
        ),
        "ndcg_sum": float(
            (
                dcg[valid_ndcg]
                / ideal_dcg[valid_ndcg]
            ).sum()
        )
    }

    result.update({
        "overall_mae": (
            result["absolute_error_sum"]
            / result["row_count"]
        ),
        "active_target_mae": (
            result[
                "active_absolute_error_sum"
            ] / result["active_count"]
            if result["active_count"] > 0
            else np.nan
        ),
        "pressure_capture_at_k": (
            result["selected_pressure"]
            / result["total_pressure"]
            if result["total_pressure"] > 0
            else 0
        ),
        "active_zone_recall_at_k": (
            result["selected_active"]
            / result["active_count"]
            if result["active_count"] > 0
            else 0
        ),
        "hotspot_recall_at_k": (
            result["selected_hotspots"]
            / result["total_hotspots"]
            if result["total_hotspots"] > 0
            else 0
        ),
        "hotspot_hit_rate_at_k": (
            result["hit_hotspot_hours"]
            / result["hotspot_hours"]
            if result["hotspot_hours"] > 0
            else 0
        ),
        "mean_ndcg_at_k": (
            result["ndcg_sum"]
            / result["valid_ndcg_hours"]
            if result["valid_ndcg_hours"] > 0
            else 0
        )
    })

    return result

In [9]:
# Train one model per held-out geographic region

spatial_features = feature_sets[
    "no_zone_identity"
]

spatial_categorical_features = [
    col for col in [
        "hour_of_day",
        "day_of_week",
        "month"
    ]
    if col in spatial_features
]

assert "zone_index" not in spatial_features
assert "rolling_mean_24h" in model_data.columns

model_data = model_data.merge(
    spatial_fold_map,
    on="zone_index",
    how="left",
    validate="many_to_one"
)

assert model_data["spatial_fold"].notna().all()

spatial_holdout_rows = []

for fold_id in range(SPATIAL_FOLDS):
    heldout_zones = spatial_fold_map.loc[
        spatial_fold_map["spatial_fold"]
        == fold_id,
        "zone_index"
    ].to_numpy()

    fold_train = model_data.loc[
        model_data["split_code"].eq(1)
        & ~model_data[
            "zone_index"
        ].isin(heldout_zones)
    ]

    positive_indices = fold_train.index[
        fold_train[
            "target_next_hour_pressure"
        ] > 0
    ].to_numpy()

    zero_indices = fold_train.index[
        fold_train[
            "target_next_hour_pressure"
        ] == 0
    ].to_numpy()

    fold_rng = np.random.default_rng(
        RANDOM_STATE + fold_id
    )

    fold_zero_count = min(
        len(zero_indices),
        NEGATIVE_TO_POSITIVE_RATIO
        * len(positive_indices)
    )

    sampled_zero_indices = fold_rng.choice(
        zero_indices,
        size=fold_zero_count,
        replace=False
    )

    fold_training_indices = np.concatenate([
        positive_indices,
        sampled_zero_indices
    ])

    fold_rng.shuffle(
        fold_training_indices
    )

    fold_train_sample = model_data.loc[
        fold_training_indices
    ]

    fold_y = fold_train_sample[
        "target_next_hour_pressure"
    ].to_numpy(dtype=np.float32)

    fold_sampling_correction = (
        len(zero_indices)
        / fold_zero_count
    )

    fold_weights = (
        np.where(
            fold_y == 0,
            fold_sampling_correction,
            1.0
        )
        * (1 + np.sqrt(fold_y))
    ).astype("float32")

    X_fold_train = fold_train_sample[
        spatial_features
    ].copy()

    for col in spatial_categorical_features:
        X_fold_train[col] = (
            X_fold_train[col].astype(str)
        )

    model_parameters = {
        "iterations": 150,
        "depth": 8,
        "learning_rate": 0.05,
        "loss_function": "RMSE",
        "l2_leaf_reg": 5,
        "random_seed":
            RANDOM_STATE + fold_id,
        "allow_writing_files": False,
        "verbose": False
    }

    if use_gpu:
        model_parameters.update({
            "task_type": "GPU",
            "devices": "0"
        })

    spatial_model = CatBoostRegressor(
        **model_parameters
    )

    print(
        f"Training fold {fold_id}: "
        f"{len(heldout_zones)} unseen zones, "
        f"{len(fold_train_sample):,} sampled rows"
    )

    spatial_model.fit(
        X_fold_train,
        np.log1p(fold_y),
        sample_weight=fold_weights,
        cat_features=(
            spatial_categorical_features
        )
    )

    del X_fold_train
    gc.collect()

    for split_code, split_name in [
        (2, "validation"),
        (3, "test")
    ]:
        fold_evaluation = (
            model_data.loc[
                model_data[
                    "split_code"
                ].eq(split_code)
                & model_data[
                    "zone_index"
                ].isin(heldout_zones)
            ]
            .sort_values([
                "target_time",
                "zone_index"
            ])
        )

        X_fold_evaluation = (
            fold_evaluation[
                spatial_features
            ].copy()
        )

        for col in (
            spatial_categorical_features
        ):
            X_fold_evaluation[col] = (
                X_fold_evaluation[
                    col
                ].astype(str)
            )

        predicted_pressure = np.clip(
            np.expm1(
                spatial_model.predict(
                    X_fold_evaluation
                )
            ),
            0,
            None
        ).astype("float32")

        n_hours = fold_evaluation[
            "target_time"
        ].nunique()

        n_zones = len(heldout_zones)

        assert (
            len(fold_evaluation)
            == n_hours * n_zones
        )

        target_matrix = (
            fold_evaluation[
                "target_next_hour_pressure"
            ]
            .to_numpy(dtype=np.float32)
            .reshape(n_hours, n_zones)
        )

        model_prediction_matrix = (
            predicted_pressure.reshape(
                n_hours,
                n_zones
            )
        )

        baseline_prediction_matrix = (
            fold_evaluation[
                "rolling_mean_24h"
            ]
            .fillna(0)
            .to_numpy(dtype=np.float32)
            .reshape(n_hours, n_zones)
        )

        proportional_k = max(
            1,
            int(
                round(
                    25
                    * n_zones
                    / 1163
                )
            )
        )

        spatial_holdout_rows.append(
            evaluate_spatial_block(
                target=target_matrix,
                prediction=(
                    model_prediction_matrix
                ),
                model_name=(
                    "spatial_holdout_catboost"
                ),
                split_name=split_name,
                fold_id=fold_id,
                top_k=proportional_k
            )
        )

        spatial_holdout_rows.append(
            evaluate_spatial_block(
                target=target_matrix,
                prediction=(
                    baseline_prediction_matrix
                ),
                model_name=(
                    "recent_24h_baseline"
                ),
                split_name=split_name,
                fold_id=fold_id,
                top_k=proportional_k
            )
        )

        del X_fold_evaluation
        gc.collect()

    del spatial_model
    del fold_train_sample
    gc.collect()

spatial_holdout_results = pd.DataFrame(
    spatial_holdout_rows
)

display(
    spatial_holdout_results[
        [
            "spatial_fold",
            "split",
            "variant",
            "heldout_zones",
            "top_k",
            "overall_mae",
            "active_target_mae",
            "pressure_capture_at_k",
            "hotspot_recall_at_k",
            "hotspot_hit_rate_at_k",
            "mean_ndcg_at_k"
        ]
    ].sort_values(
        ["split", "spatial_fold", "variant"]
    )
)

Training fold 0: 219 unseen zones, 613,535 sampled rows
Training fold 1: 285 unseen zones, 519,818 sampled rows
Training fold 2: 201 unseen zones, 577,434 sampled rows
Training fold 3: 78 unseen zones, 654,888 sampled rows
Training fold 4: 380 unseen zones, 396,513 sampled rows


,spatial_fold,split,variant,heldout_zones,top_k,overall_mae,active_target_mae,pressure_capture_at_k,hotspot_recall_at_k,hotspot_hit_rate_at_k,mean_ndcg_at_k
3,0,test,recent_24h_baseline,219,5,0.0605,2.7612,0.1159,0.1532,0.1932,0.0698
2,0,test,spatial_holdout_catboost,219,5,0.0650,2.6668,0.2220,0.2703,0.3295,0.1717
7,1,test,recent_24h_baseline,285,6,0.1158,3.1473,0.1921,0.2278,0.3459,0.1804
6,1,test,spatial_holdout_catboost,285,6,0.1109,2.9258,0.2684,0.3254,0.4919,0.2661
11,2,test,recent_24h_baseline,201,4,0.0922,2.9686,0.2104,0.3151,0.3947,0.1743
10,2,test,spatial_holdout_catboost,201,4,0.0923,2.7846,0.3384,0.4521,0.5439,0.3330
15,3,test,recent_24h_baseline,78,2,0.0930,4.2625,0.3920,0.4337,0.4932,0.2926
14,3,test,spatial_holdout_catboost,78,2,0.0888,3.9352,0.5655,0.6627,0.7397,0.4273
19,4,test,recent_24h_baseline,380,8,0.1457,3.2516,0.3231,0.4059,0.6055,0.2816
18,4,test,spatial_holdout_catboost,380,8,0.1321,2.9821,0.3982,0.5174,0.7305,0.4051


In [10]:
# Aggregate spatial performance across folds

# Numerators and denominators are pooled across all five held-out blocks. This produces the overall result for zones

aggregate_rows = []

for (
    split_name,
    variant
), group in spatial_holdout_results.groupby(
    ["split", "variant"]
):
    totals = group[
        [
            "row_count",
            "active_count",
            "absolute_error_sum",
            "active_absolute_error_sum",
            "total_pressure",
            "selected_pressure",
            "selected_active",
            "total_hotspots",
            "selected_hotspots",
            "hotspot_hours",
            "hit_hotspot_hours",
            "valid_ndcg_hours",
            "ndcg_sum"
        ]
    ].sum()

    aggregate_rows.append({
        "split": split_name,
        "variant": variant,
        "heldout_zones": 1163,

        "overall_mae": (
            totals["absolute_error_sum"]
            / totals["row_count"]
        ),

        "active_target_mae": (
            totals[
                "active_absolute_error_sum"
            ]
            / totals["active_count"]
        ),

        "pressure_capture_at_proportional_k": (
            totals["selected_pressure"]
            / totals["total_pressure"]
        ),

        "active_zone_recall_at_proportional_k": (
            totals["selected_active"]
            / totals["active_count"]
        ),

        "hotspot_recall_at_proportional_k": (
            totals["selected_hotspots"]
            / totals["total_hotspots"]
        ),

        "hotspot_hit_rate_at_proportional_k": (
            totals["hit_hotspot_hours"]
            / totals["hotspot_hours"]
        ),

        "mean_ndcg_at_proportional_k": (
            totals["ndcg_sum"]
            / totals["valid_ndcg_hours"]
        )
    })

spatial_holdout_summary = pd.DataFrame(
    aggregate_rows
)

display(spatial_holdout_summary)

spatial_holdout_results.to_csv(
    OUTPUT_DIR
    / "spatial_holdout_fold_results.csv",
    index=False
)

spatial_holdout_summary.to_csv(
    OUTPUT_DIR
    / "spatial_holdout_summary.csv",
    index=False
)

spatial_fold_summary.to_csv(
    OUTPUT_DIR
    / "spatial_holdout_regions.csv",
    index=False
)

print("Spatial holdout outputs saved.")

,split,variant,heldout_zones,overall_mae,active_target_mae,pressure_capture_at_proportional_k,active_zone_recall_at_proportional_k,hotspot_recall_at_proportional_k,hotspot_hit_rate_at_proportional_k,mean_ndcg_at_proportional_k
0,test,recent_24h_baseline,1163,0.1095,3.1655,0.2575,0.1550,0.3267,0.4427,0.1958
1,test,spatial_holdout_catboost,1163,0.1045,2.9398,0.3488,0.2180,0.4457,0.5908,0.3164
2,validation,recent_24h_baseline,1163,0.1113,3.2167,0.2360,0.1367,0.3049,0.4180,0.1690
3,validation,spatial_holdout_catboost,1163,0.1065,3.0227,0.3285,0.2101,0.4136,0.5451,0.2861


Spatial holdout outputs saved.


## Threshold and Confidence Robustness

In [11]:
# load frozen prediction and algint them with model table

PREDICTION_PATH = find_input_file(
    "calibrated_pressure_predictions.parquet"
)

prediction_columns = [
    "zone_index",
    "target_time",
    "actual_pressure",
    "predicted_pressure",
    "calibrated_pressure",
    "hotspot_probability"
]

frozen_predictions = pd.read_parquet(
    PREDICTION_PATH,
    columns=prediction_columns
)

reference_columns = [
    "zone_index",
    "target_time",
    "split_code",
    "target_next_hour_pressure",
    "rolling_mean_24h"
]

prediction_reference = (
    model_data.loc[
        model_data["split_code"].isin([2, 3]),
        reference_columns
    ]
    .copy()
)

robustness_panel = prediction_reference.merge(
    frozen_predictions,
    on=["zone_index", "target_time"],
    how="inner",
    validate="one_to_one"
)

robustness_panel = robustness_panel.sort_values(
    ["split_code", "target_time", "zone_index"]
).reset_index(drop=True)

assert np.allclose(
    robustness_panel["target_next_hour_pressure"],
    robustness_panel["actual_pressure"]
)

assert len(robustness_panel) == 1_265_344

print("Robustness rows:", len(robustness_panel))
print(
    "Hours:",
    robustness_panel["target_time"].nunique()
)
print(
    "Zones:",
    robustness_panel["zone_index"].nunique()
)

Robustness rows: 1265344
Hours: 1088
Zones: 1163


In [12]:
# Evaluate multiple hotspot definitions

# The primary study used eight incidents as the hotspot threshold. 
# This cell repeats the ranking evaluation at thresholds 4, 6, 8, 10 and 12 and capacities 5, 10 and 25

HOTSPOT_THRESHOLDS = [4, 6, 8, 10, 12]
ROBUSTNESS_K_VALUES = [5, 10, 25]

ranking_sensitivity_rows = []

for split_code, split_name in [
    (2, "validation"),
    (3, "test")
]:
    split_data = (
        robustness_panel.loc[
            robustness_panel["split_code"]
            == split_code
        ]
        .sort_values([
            "target_time",
            "zone_index"
        ])
    )

    n_hours = split_data[
        "target_time"
    ].nunique()

    n_zones = split_data[
        "zone_index"
    ].nunique()

    actual_matrix = (
        split_data["actual_pressure"]
        .to_numpy(dtype=np.float32)
        .reshape(n_hours, n_zones)
    )

    score_variants = {
        "frozen_catboost":
            split_data["predicted_pressure"]
            .to_numpy(dtype=np.float32)
            .reshape(n_hours, n_zones),

        "recent_24h_baseline":
            split_data["rolling_mean_24h"]
            .fillna(0)
            .to_numpy(dtype=np.float32)
            .reshape(n_hours, n_zones)
    }

    for variant, score_matrix in (
        score_variants.items()
    ):
        complete_ranking = np.argsort(
            -score_matrix,
            axis=1,
            kind="stable"
        )

        for k in ROBUSTNESS_K_VALUES:
            top_indices = complete_ranking[:, :k]

            selected_actual = np.take_along_axis(
                actual_matrix,
                top_indices,
                axis=1
            )

            ideal_actual = np.sort(
                actual_matrix,
                axis=1
            )[:, ::-1][:, :k]

            discounts = (
                1
                / np.log2(
                    np.arange(2, k + 2)
                )
            ).astype("float32")

            dcg = (
                selected_actual
                * discounts
            ).sum(axis=1)

            ideal_dcg = (
                ideal_actual
                * discounts
            ).sum(axis=1)

            valid_ndcg = ideal_dcg > 0

            pressure_capture = (
                selected_actual.sum()
                / actual_matrix.sum()
            )

            active_recall = (
                (selected_actual > 0).sum()
                / (actual_matrix > 0).sum()
            )

            mean_ndcg = (
                (
                    dcg[valid_ndcg]
                    / ideal_dcg[valid_ndcg]
                ).mean()
            )

            for hotspot_threshold in (
                HOTSPOT_THRESHOLDS
            ):
                true_hotspots = (
                    actual_matrix
                    >= hotspot_threshold
                )

                selected_hotspots = (
                    selected_actual
                    >= hotspot_threshold
                )

                hotspot_hours = (
                    true_hotspots.any(axis=1)
                )

                ranking_sensitivity_rows.append({
                    "split": split_name,
                    "variant": variant,
                    "k": k,
                    "hotspot_threshold":
                        hotspot_threshold,
                    "total_hotspots":
                        int(true_hotspots.sum()),
                    "hotspot_hours":
                        int(hotspot_hours.sum()),
                    "pressure_capture_at_k":
                        pressure_capture,
                    "active_zone_recall_at_k":
                        active_recall,
                    "hotspot_recall_at_k": (
                        selected_hotspots.sum()
                        / true_hotspots.sum()
                        if true_hotspots.sum() > 0
                        else 0
                    ),
                    "hotspot_hit_rate_at_k": (
                        selected_hotspots[
                            hotspot_hours
                        ].any(axis=1).mean()
                        if hotspot_hours.any()
                        else 0
                    ),
                    "mean_ndcg_at_k":
                        mean_ndcg
                })

ranking_sensitivity_results = pd.DataFrame(
    ranking_sensitivity_rows
)

display(
    ranking_sensitivity_results.sort_values(
        [
            "split",
            "k",
            "hotspot_threshold",
            "variant"
        ]
    )
)

,split,variant,k,hotspot_threshold,total_hotspots,hotspot_hours,pressure_capture_at_k,active_zone_recall_at_k,hotspot_recall_at_k,hotspot_hit_rate_at_k,mean_ndcg_at_k
30,test,frozen_catboost,5,4,3073,377,0.1774,0.0861,0.1715,0.7586,0.3293
45,test,recent_24h_baseline,5,4,3073,377,0.0956,0.0479,0.0882,0.4881,0.1503
31,test,frozen_catboost,5,6,1883,358,0.1774,0.0861,0.2055,0.6732,0.3293
46,test,recent_24h_baseline,5,6,1883,358,0.0956,0.0479,0.1105,0.4246,0.1503
32,test,frozen_catboost,5,8,1252,341,0.1774,0.0861,0.2396,0.5924,0.3293
47,test,recent_24h_baseline,5,8,1252,341,0.0956,0.0479,0.1286,0.3724,0.1503
33,test,frozen_catboost,5,10,843,310,0.1774,0.0861,0.2716,0.5710,0.3293
48,test,recent_24h_baseline,5,10,843,310,0.0956,0.0479,0.1400,0.3323,0.1503
34,test,frozen_catboost,5,12,587,274,0.1774,0.0861,0.3152,0.5401,0.3293
49,test,recent_24h_baseline,5,12,587,274,0.0956,0.0479,0.1687,0.3248,0.1503


In [13]:
# Compare the model directly against the baseline
model_sensitivity = (
    ranking_sensitivity_results.loc[
        ranking_sensitivity_results[
            "variant"
        ] == "frozen_catboost"
    ]
    .drop(columns="variant")
)

baseline_sensitivity = (
    ranking_sensitivity_results.loc[
        ranking_sensitivity_results[
            "variant"
        ] == "recent_24h_baseline"
    ]
    .drop(columns="variant")
)

sensitivity_comparison = (
    model_sensitivity.merge(
        baseline_sensitivity,
        on=[
            "split",
            "k",
            "hotspot_threshold"
        ],
        suffixes=(
            "_model",
            "_baseline"
        ),
        validate="one_to_one"
    )
)

comparison_metrics = [
    "pressure_capture_at_k",
    "active_zone_recall_at_k",
    "hotspot_recall_at_k",
    "hotspot_hit_rate_at_k",
    "mean_ndcg_at_k"
]

for metric in comparison_metrics:
    sensitivity_comparison[
        f"{metric}_absolute_gain"
    ] = (
        sensitivity_comparison[
            f"{metric}_model"
        ]
        - sensitivity_comparison[
            f"{metric}_baseline"
        ]
    )

    sensitivity_comparison[
        f"{metric}_relative_gain_percent"
    ] = (
        100
        * sensitivity_comparison[
            f"{metric}_absolute_gain"
        ]
        / sensitivity_comparison[
            f"{metric}_baseline"
        ].replace(0, np.nan)
    )

display(
    sensitivity_comparison.loc[
        sensitivity_comparison["split"]
        == "test",
        [
            "k",
            "hotspot_threshold",
            "total_hotspots_model",
            "hotspot_recall_at_k_model",
            "hotspot_recall_at_k_baseline",
            "hotspot_recall_at_k_relative_gain_percent",
            "hotspot_hit_rate_at_k_model",
            "hotspot_hit_rate_at_k_baseline",
            "pressure_capture_at_k_model",
            "pressure_capture_at_k_baseline",
            "mean_ndcg_at_k_model",
            "mean_ndcg_at_k_baseline"
        ]
    ].sort_values(
        ["k", "hotspot_threshold"]
    )
)

,k,hotspot_threshold,total_hotspots_model,hotspot_recall_at_k_model,hotspot_recall_at_k_baseline,hotspot_recall_at_k_relative_gain_percent,hotspot_hit_rate_at_k_model,hotspot_hit_rate_at_k_baseline,pressure_capture_at_k_model,pressure_capture_at_k_baseline,mean_ndcg_at_k_model,mean_ndcg_at_k_baseline
15,5,4,3073,0.1715,0.0882,94.4649,0.7586,0.4881,0.1774,0.0956,0.3293,0.1503
16,5,6,1883,0.2055,0.1105,86.0577,0.6732,0.4246,0.1774,0.0956,0.3293,0.1503
17,5,8,1252,0.2396,0.1286,86.3354,0.5924,0.3724,0.1774,0.0956,0.3293,0.1503
18,5,10,843,0.2716,0.1400,94.0678,0.5710,0.3323,0.1774,0.0956,0.3293,0.1503
19,5,12,587,0.3152,0.1687,86.8687,0.5401,0.3248,0.1774,0.0956,0.3293,0.1503
20,10,4,3073,0.2584,0.1588,62.7049,0.8647,0.6684,0.2602,0.1639,0.3512,0.1777
21,10,6,1883,0.3016,0.1880,60.4520,0.7849,0.5950,0.2602,0.1639,0.3512,0.1777
22,10,8,1252,0.3474,0.2165,60.5166,0.7214,0.5337,0.2602,0.1639,0.3512,0.1777
23,10,10,843,0.3903,0.2384,63.6816,0.6935,0.4839,0.2602,0.1639,0.3512,0.1777
24,10,12,587,0.4242,0.2811,50.9091,0.6277,0.4708,0.2602,0.1639,0.3512,0.1777


In [14]:
# Audit confidence and observation-bias subgroups

# Confidence is not used to suppress the forecast. 
# This audit checks whether low-confidence and potentially biased observation groups have systematically different errors
audit_columns = [
    "zone_index",
    "target_time",
    "split_code",
    "observation_confidence",
    "observer_bias_penalty",
    "cold_start_zone"
]

available_audit_columns = [
    col for col in audit_columns
    if col in schema_columns
]

confidence_data = pd.read_parquet(
    MODEL_TABLE_PATH,
    columns=available_audit_columns,
    filters=[
        ("split_code", "in", [2, 3])
    ]
)

confidence_panel = robustness_panel.merge(
    confidence_data,
    on=[
        "zone_index",
        "target_time",
        "split_code"
    ],
    how="left",
    validate="one_to_one"
)

LOW_CONFIDENCE_THRESHOLD = (
    0.7553357481956482
)

HIGH_CONFIDENCE_THRESHOLD = (
    0.9071022272109985
)

confidence_panel["confidence_band"] = np.select(
    [
        confidence_panel[
            "observation_confidence"
        ] < LOW_CONFIDENCE_THRESHOLD,

        confidence_panel[
            "observation_confidence"
        ] >= HIGH_CONFIDENCE_THRESHOLD
    ],
    [
        "low_confidence",
        "high_confidence"
    ],
    default="medium_confidence"
)

confidence_panel["pressure_error"] = (
    confidence_panel["calibrated_pressure"]
    - confidence_panel["actual_pressure"]
)

confidence_panel["absolute_error"] = (
    confidence_panel["pressure_error"].abs()
)

confidence_panel["squared_hotspot_error"] = (
    confidence_panel["hotspot_probability"]
    - (
        confidence_panel["actual_pressure"]
        >= 8
    ).astype("int8")
) ** 2

confidence_audit = (
    confidence_panel.groupby(
        ["split_code", "confidence_band"],
        observed=True
    )
    .agg(
        rows=("zone_index", "size"),
        zones=("zone_index", "nunique"),
        mean_actual_pressure=(
            "actual_pressure",
            "mean"
        ),
        active_rate=(
            "actual_pressure",
            lambda x: (x > 0).mean()
        ),
        hotspot_rate=(
            "actual_pressure",
            lambda x: (x >= 8).mean()
        ),
        mean_calibrated_pressure=(
            "calibrated_pressure",
            "mean"
        ),
        pressure_bias=(
            "pressure_error",
            "mean"
        ),
        overall_mae=(
            "absolute_error",
            "mean"
        ),
        hotspot_brier_score=(
            "squared_hotspot_error",
            "mean"
        ),
        mean_observer_bias_penalty=(
            "observer_bias_penalty",
            "mean"
        ),
        cold_start_rows=(
            "cold_start_zone",
            "sum"
        )
    )
    .reset_index()
)

confidence_audit["split"] = (
    confidence_audit["split_code"]
    .map({
        2: "validation",
        3: "test"
    })
)

active_confidence_audit = (
    confidence_panel.loc[
        confidence_panel[
            "actual_pressure"
        ] > 0
    ]
    .groupby(
        ["split_code", "confidence_band"],
        observed=True
    )["absolute_error"]
    .mean()
    .rename("active_target_mae")
    .reset_index()
)

confidence_audit = confidence_audit.merge(
    active_confidence_audit,
    on=["split_code", "confidence_band"],
    how="left",
    validate="one_to_one"
)

display(
    confidence_audit[
        [
            "split",
            "confidence_band",
            "rows",
            "zones",
            "mean_actual_pressure",
            "active_rate",
            "hotspot_rate",
            "mean_calibrated_pressure",
            "pressure_bias",
            "overall_mae",
            "active_target_mae",
            "hotspot_brier_score",
            "mean_observer_bias_penalty",
            "cold_start_rows"
        ]
    ].sort_values(
        ["split", "confidence_band"]
    )
)

,split,confidence_band,rows,zones,mean_actual_pressure,active_rate,hotspot_rate,mean_calibrated_pressure,pressure_bias,overall_mae,active_target_mae,hotspot_brier_score,mean_observer_bias_penalty,cold_start_rows
3,test,high_confidence,39695,73,0.3290,0.0754,0.0119,0.3299,0.0009,0.4665,3.3399,0.0105,0.0877,0
4,test,low_confidence,324241,612,0.0246,0.0077,0.0007,0.0230,-0.0015,0.0437,2.9499,0.0007,0.5637,1088
5,test,medium_confidence,268736,503,0.0647,0.0206,0.0020,0.0643,-0.0003,0.1081,2.6858,0.0019,0.2223,0
0,validation,high_confidence,39569,73,0.3075,0.0717,0.0113,0.3128,0.0053,0.4554,3.3747,0.0103,0.0877,0
1,validation,low_confidence,324366,689,0.0258,0.0081,0.0008,0.0247,-0.0011,0.0462,2.9415,0.0007,0.5635,1088
2,validation,medium_confidence,268737,535,0.0662,0.0204,0.0020,0.0668,0.0006,0.1123,2.8107,0.0019,0.2223,0


In [15]:
# save outputs
ranking_sensitivity_results.to_csv(
    OUTPUT_DIR
    / "hotspot_threshold_sensitivity.csv",
    index=False
)

sensitivity_comparison.to_csv(
    OUTPUT_DIR
    / "hotspot_threshold_model_vs_baseline.csv",
    index=False
)

confidence_audit.to_csv(
    OUTPUT_DIR
    / "confidence_subgroup_audit.csv",
    index=False
)

print("Threshold and confidence audits saved.")

Threshold and confidence audits saved.


## Prototype Package and Consolidated Evaluation

In [16]:
# load and validate outputs

from pathlib import Path
import json
import zipfile



PHASE6_RECOMMENDATIONS_PATH = find_input_file(
    "final_adaptive_patrol_recommendations.parquet"
)

PHASE6_SCHEDULE_PATH = find_input_file(
    "adaptive_hourly_dispatch_schedule.csv"
)

PHASE6_EVALUATION_PATH = find_input_file(
    "adaptive_dispatch_evaluation.csv"
)

PHASE6_THRESHOLD_PATH = find_input_file(
    "adaptive_dispatch_threshold.csv"
)

adaptive_recommendations = pd.read_parquet(
    PHASE6_RECOMMENDATIONS_PATH
)

adaptive_schedule = pd.read_csv(
    PHASE6_SCHEDULE_PATH
)

adaptive_dispatch_evaluation = pd.read_csv(
    PHASE6_EVALUATION_PATH
)

dispatch_threshold_table = pd.read_csv(
    PHASE6_THRESHOLD_PATH
)

adaptive_recommendations["target_time"] = (
    pd.to_datetime(
        adaptive_recommendations["target_time"],
        utc=True
    )
    .dt.tz_convert("Asia/Kolkata")
)

adaptive_schedule["target_time"] = (
    pd.to_datetime(
        adaptive_schedule["target_time"],
        utc=True
    )
    .dt.tz_convert("Asia/Kolkata")
)

required_columns = [
    "zone_index",
    "target_time",
    "selection_order",
    "latitude",
    "longitude",
    "spatial_priority_score",
    "calibrated_pressure",
    "hotspot_probability",
    "observation_confidence",
    "confidence_tier",
    "patrol_action",
    "evidence_summary",
    "local_pressure_removed_30pct",
    "local_pressure_removed_60pct",
    "local_pressure_removed_90pct"
]

missing_columns = [
    col for col in required_columns
    if col not in adaptive_recommendations.columns
]

if missing_columns:
    raise KeyError(
        "Missing recommendation columns: "
        f"{missing_columns}"
    )

assert not adaptive_recommendations.duplicated(
    ["target_time", "zone_index"]
).any()

assert adaptive_recommendations[
    "selection_order"
].between(1, 25).all()

MIN_DISPATCH_SCORE = float(
    dispatch_threshold_table.iloc[0][
        "minimum_dispatch_score"
    ]
)

print(
    "Recommendation rows:",
    len(adaptive_recommendations)
)

print(
    "Hours represented:",
    adaptive_recommendations[
        "target_time"
    ].nunique()
)

print(
    "Minimum dispatch score:",
    MIN_DISPATCH_SCORE
)

print(
    "Recommendation score minimum:",
    adaptive_recommendations[
        "spatial_priority_score"
    ].min()
)

Recommendation rows: 15021
Hours represented: 831
Minimum dispatch score: 0.5886576771736145
Recommendation score minimum: 0.5887099504470825


In [17]:
# creating dispatch tables

def create_reason_codes(row):
    codes = []

    if row.get("incident_count", 0) > 0:
        codes.append("CURRENT_INCIDENT_ACTIVITY")

    if row.get("current_burst_z", 0) >= 2:
        codes.append("TEMPORAL_BURST")

    if row.get("relative_to_same_hour", 0) >= 2:
        codes.append("ABOVE_USUAL_SAME_HOUR")

    if row.get(
        "consecutive_active_hours_before",
        0
    ) >= 2:
        codes.append("PERSISTENT_RECENT_ACTIVITY")

    if row.get("active_neighbour_count", 0) >= 2:
        codes.append("NEARBY_ACTIVITY")

    if row.get(
        "explicit_road_obstruction",
        0
    ) == 1:
        codes.append("ROAD_INTERFACE_EVIDENCE")

    if row.get(
        "large_vehicle_evidence",
        0
    ) == 1:
        codes.append("LARGE_VEHICLE_EVIDENCE")

    if row.get(
        "multi_offence_evidence",
        0
    ) == 1:
        codes.append("MULTIPLE_OFFENCE_EVIDENCE")

    if row.get(
        "confidence_tier",
        ""
    ) == "low_confidence":
        codes.append("ON_SITE_VERIFICATION_REQUIRED")

    if not codes:
        codes.append("FORECAST_RECURRENCE_SIGNAL")

    return "|".join(codes)


prototype_dispatch = adaptive_recommendations.copy()

prototype_dispatch = prototype_dispatch.merge(
    adaptive_schedule[
        [
            "target_time",
            "recommended_patrols",
            "deployment_status"
        ]
    ],
    on="target_time",
    how="left",
    validate="many_to_one"
)

prototype_dispatch[
    "recommendation_id"
] = (
    prototype_dispatch["target_time"]
    .dt.strftime("%Y%m%dT%H%M%S%z")
    + "_Z"
    + prototype_dispatch[
        "zone_index"
    ].astype(str)
)

prototype_dispatch["dispatch_tier"] = np.select(
    [
        prototype_dispatch[
            "selection_order"
        ] <= 5,

        prototype_dispatch[
            "selection_order"
        ] <= 10
    ],
    [
        "highest_priority",
        "secondary_priority"
    ],
    default="extended_coverage"
)

prototype_dispatch[
    "verification_required"
] = (
    prototype_dispatch[
        "confidence_tier"
    ].eq("low_confidence")
    | prototype_dispatch[
        "patrol_action"
    ].eq("verify_before_enforcement")
)

prototype_dispatch[
    "has_current_incident_evidence"
] = (
    prototype_dispatch[
        "incident_count"
    ].gt(0)
)

prototype_dispatch["reason_codes"] = (
    prototype_dispatch.apply(
        create_reason_codes,
        axis=1
    )
)

prototype_dispatch[
    "hotspot_probability_percent"
] = (
    100
    * prototype_dispatch[
        "hotspot_probability"
    ]
)

prototype_dispatch[
    "scenario_scope"
] = (
    "hypothetical local pressure suppression; "
    "not a causal treatment estimate"
)

prototype_dispatch[
    "spatial_context_role"
] = (
    "neighbourhood ranking and patrol coverage; "
    "not learned congestion propagation"
)

prototype_dispatch[
    "dispatch_threshold_score"
] = MIN_DISPATCH_SCORE

prototype_dispatch[
    "patrol_balance_alpha"
] = 0.60

prototype_dispatch[
    "coverage_radius_m"
] = 500


prototype_columns = [
    "recommendation_id",
    "target_time",
    "split",
    "selection_order",
    "dispatch_tier",
    "recommended_patrols",
    "deployment_status",
    "zone_index",
    "zone_250m",
    "latitude",
    "longitude",
    "spatial_priority_score",
    "calibrated_pressure",
    "hotspot_probability",
    "hotspot_probability_percent",
    "incident_count",
    "unique_vehicles",
    "distance_neighbour_exposure",
    "active_neighbour_count",
    "current_burst_z",
    "relative_to_same_hour",
    "consecutive_active_hours_before",
    "explicit_road_obstruction",
    "large_vehicle_evidence",
    "multi_offence_evidence",
    "disruption_class",
    "observation_confidence",
    "confidence_tier",
    "verification_required",
    "has_current_incident_evidence",
    "patrol_action",
    "reason_codes",
    "evidence_summary",
    "confidence_note",
    "local_pressure_removed_30pct",
    "local_pressure_removed_60pct",
    "local_pressure_removed_90pct",
    "dispatch_threshold_score",
    "patrol_balance_alpha",
    "coverage_radius_m",
    "scenario_scope",
    "spatial_context_role"
]

prototype_columns = [
    col for col in prototype_columns
    if col in prototype_dispatch.columns
]

prototype_dispatch = prototype_dispatch[
    prototype_columns
].sort_values(
    ["target_time", "selection_order"]
).reset_index(drop=True)

assert prototype_dispatch[
    "recommendation_id"
].is_unique

assert prototype_dispatch[
    "spatial_priority_score"
].ge(MIN_DISPATCH_SCORE).all()

latest_target_time = prototype_dispatch[
    "target_time"
].max()

latest_prototype_dispatch = (
    prototype_dispatch.loc[
        prototype_dispatch[
            "target_time"
        ] == latest_target_time
    ]
    .copy()
)

print("Prototype rows:", len(prototype_dispatch))
print("Prototype columns:", len(prototype_dispatch.columns))
print("Latest target time:", latest_target_time)
print(
    "Latest patrol count:",
    len(latest_prototype_dispatch)
)

display(
    latest_prototype_dispatch[
        [
            "selection_order",
            "dispatch_tier",
            "zone_index",
            "latitude",
            "longitude",
            "spatial_priority_score",
            "calibrated_pressure",
            "hotspot_probability_percent",
            "observation_confidence",
            "patrol_action",
            "reason_codes"
        ]
    ]
)

Prototype rows: 15021
Prototype columns: 42
Latest target time: 2024-04-08 23:00:00+05:30
Latest patrol count: 3


,selection_order,dispatch_tier,zone_index,latitude,longitude,spatial_priority_score,calibrated_pressure,hotspot_probability_percent,observation_confidence,patrol_action,reason_codes
15018,1,highest_priority,401,12.9643,77.5769,0.7510,0.6709,2.1235,0.9100,preventive_patrol,FORECAST_RECURRENCE_SIGNAL
15019,2,highest_priority,1062,13.0082,77.6954,0.7389,0.5693,2.1235,0.8963,preventive_patrol,FORECAST_RECURRENCE_SIGNAL
15020,3,highest_priority,371,12.9647,77.5758,0.6462,0.5527,1.9369,0.9199,targeted_obstruction_enforcement,CURRENT_INCIDENT_ACTIVITY|ROAD_INTERFACE_EVIDE...


In [18]:
# Build the consolidated evaluation summary

if "sensitivity_comparison" not in globals():
    sensitivity_comparison = pd.read_csv(
        OUTPUT_DIR
        / "hotspot_threshold_model_vs_baseline.csv"
    )

if "spatial_holdout_summary" not in globals():
    spatial_holdout_summary = pd.read_csv(
        OUTPUT_DIR
        / "spatial_holdout_summary.csv"
    )

if "test_ablation_comparison" not in globals():
    test_ablation_comparison = pd.read_csv(
        OUTPUT_DIR
        / "forecast_test_ablation_comparison.csv"
    )

if "confidence_audit" not in globals():
    confidence_audit = pd.read_csv(
        OUTPUT_DIR
        / "confidence_subgroup_audit.csv"
    )


official_result = sensitivity_comparison.loc[
    sensitivity_comparison[
        "split"
    ].eq("test")
    & sensitivity_comparison[
        "k"
    ].eq(25)
    & sensitivity_comparison[
        "hotspot_threshold"
    ].eq(8)
].iloc[0]

spatial_model_result = (
    spatial_holdout_summary.loc[
        spatial_holdout_summary[
            "split"
        ].eq("test")
        & spatial_holdout_summary[
            "variant"
        ].eq("spatial_holdout_catboost")
    ].iloc[0]
)

spatial_baseline_result = (
    spatial_holdout_summary.loc[
        spatial_holdout_summary[
            "split"
        ].eq("test")
        & spatial_holdout_summary[
            "variant"
        ].eq("recent_24h_baseline")
    ].iloc[0]
)

full_ablation_result = (
    test_ablation_comparison.loc[
        test_ablation_comparison[
            "model"
        ].eq("full_model")
    ].iloc[0]
)

static_ablation_result = (
    test_ablation_comparison.loc[
        test_ablation_comparison[
            "model"
        ].eq("static_heatmap_like")
    ].iloc[0]
)

no_recurrence_result = (
    test_ablation_comparison.loc[
        test_ablation_comparison[
            "model"
        ].eq("no_temporal_recurrence")
    ].iloc[0]
)

adaptive_test_result = (
    adaptive_dispatch_evaluation.loc[
        adaptive_dispatch_evaluation[
            "period"
        ].eq("test")
        & adaptive_dispatch_evaluation[
            "strategy"
        ].eq("adaptive_dispatch")
        & adaptive_dispatch_evaluation[
            "k_capacity"
        ].eq(25)
    ].iloc[0]
)

fixed_test_result = (
    adaptive_dispatch_evaluation.loc[
        adaptive_dispatch_evaluation[
            "period"
        ].eq("test")
        & adaptive_dispatch_evaluation[
            "strategy"
        ].eq("fixed_capacity")
        & adaptive_dispatch_evaluation[
            "k_capacity"
        ].eq(25)
    ].iloc[0]
)

test_confidence = confidence_audit.loc[
    confidence_audit["split"].eq("test")
]

maximum_absolute_confidence_bias = (
    test_confidence[
        "pressure_bias"
    ].abs().max()
)

test_hours = adaptive_schedule.loc[
    adaptive_schedule[
        "split_code"
    ].eq(3),
    "target_time"
].nunique()

test_scenario_gain_60 = (
    adaptive_recommendations.loc[
        adaptive_recommendations[
            "split_code"
        ].eq(3),
        "local_pressure_removed_60pct"
    ].sum()
)

headline_metrics = pd.DataFrame([
    {
        "section": "Frozen forecast",
        "metric": "Pressure capture at K=25",
        "system_value":
            official_result[
                "pressure_capture_at_k_model"
            ],
        "comparator_value":
            official_result[
                "pressure_capture_at_k_baseline"
            ],
        "relative_change_percent":
            official_result[
                "pressure_capture_at_k_relative_gain_percent"
            ],
        "comparison":
            "Recent-24-hour baseline"
    },
    {
        "section": "Frozen forecast",
        "metric": "Hotspot recall at K=25 and threshold=8",
        "system_value":
            official_result[
                "hotspot_recall_at_k_model"
            ],
        "comparator_value":
            official_result[
                "hotspot_recall_at_k_baseline"
            ],
        "relative_change_percent":
            official_result[
                "hotspot_recall_at_k_relative_gain_percent"
            ],
        "comparison":
            "Recent-24-hour baseline"
    },
    {
        "section": "Frozen forecast",
        "metric": "NDCG at K=25",
        "system_value":
            official_result[
                "mean_ndcg_at_k_model"
            ],
        "comparator_value":
            official_result[
                "mean_ndcg_at_k_baseline"
            ],
        "relative_change_percent": (
            100
            * (
                official_result[
                    "mean_ndcg_at_k_model"
                ]
                / official_result[
                    "mean_ndcg_at_k_baseline"
                ]
                - 1
            )
        ),
        "comparison":
            "Recent-24-hour baseline"
    },
    {
        "section": "Spatial holdout",
        "metric": "Pressure capture at proportional K",
        "system_value":
            spatial_model_result[
                "pressure_capture_at_proportional_k"
            ],
        "comparator_value":
            spatial_baseline_result[
                "pressure_capture_at_proportional_k"
            ],
        "relative_change_percent": (
            100
            * (
                spatial_model_result[
                    "pressure_capture_at_proportional_k"
                ]
                / spatial_baseline_result[
                    "pressure_capture_at_proportional_k"
                ]
                - 1
            )
        ),
        "comparison":
            "Unseen-zone recent-24-hour baseline"
    },
    {
        "section": "Spatial holdout",
        "metric": "Hotspot recall at proportional K",
        "system_value":
            spatial_model_result[
                "hotspot_recall_at_proportional_k"
            ],
        "comparator_value":
            spatial_baseline_result[
                "hotspot_recall_at_proportional_k"
            ],
        "relative_change_percent": (
            100
            * (
                spatial_model_result[
                    "hotspot_recall_at_proportional_k"
                ]
                / spatial_baseline_result[
                    "hotspot_recall_at_proportional_k"
                ]
                - 1
            )
        ),
        "comparison":
            "Unseen-zone recent-24-hour baseline"
    },
    {
        "section": "Temporal ablation",
        "metric": "Pressure capture versus static heatmap",
        "system_value":
            full_ablation_result[
                "pressure_capture_at_25"
            ],
        "comparator_value":
            static_ablation_result[
                "pressure_capture_at_25"
            ],
        "relative_change_percent": (
            100
            * (
                full_ablation_result[
                    "pressure_capture_at_25"
                ]
                / static_ablation_result[
                    "pressure_capture_at_25"
                ]
                - 1
            )
        ),
        "comparison":
            "Static heatmap-like model"
    },
    {
        "section": "Temporal ablation",
        "metric": "Pressure capture versus no recurrence",
        "system_value":
            full_ablation_result[
                "pressure_capture_at_25"
            ],
        "comparator_value":
            no_recurrence_result[
                "pressure_capture_at_25"
            ],
        "relative_change_percent": (
            100
            * (
                full_ablation_result[
                    "pressure_capture_at_25"
                ]
                / no_recurrence_result[
                    "pressure_capture_at_25"
                ]
                - 1
            )
        ),
        "comparison":
            "No-temporal-recurrence model"
    },
    {
        "section": "Adaptive dispatch",
        "metric": "Mean patrols deployed at capacity 25",
        "system_value":
            adaptive_test_result[
                "mean_patrols_deployed"
            ],
        "comparator_value":
            fixed_test_result[
                "mean_patrols_deployed"
            ],
        "relative_change_percent": (
            100
            * (
                adaptive_test_result[
                    "mean_patrols_deployed"
                ]
                / fixed_test_result[
                    "mean_patrols_deployed"
                ]
                - 1
            )
        ),
        "comparison":
            "Forced fixed-capacity deployment"
    },
    {
        "section": "Adaptive dispatch",
        "metric": "Direct pressure capture retained",
        "system_value":
            adaptive_test_result[
                "direct_pressure_capture"
            ],
        "comparator_value":
            fixed_test_result[
                "direct_pressure_capture"
            ],
        "relative_change_percent": (
            100
            * adaptive_test_result[
                "direct_pressure_capture"
            ]
            / fixed_test_result[
                "direct_pressure_capture"
            ]
        ),
        "comparison":
            "Percentage retained versus fixed capacity"
    },
    {
        "section": "Adaptive dispatch",
        "metric": "Area hotspot coverage retained",
        "system_value":
            adaptive_test_result[
                "area_hotspot_coverage"
            ],
        "comparator_value":
            fixed_test_result[
                "area_hotspot_coverage"
            ],
        "relative_change_percent": (
            100
            * adaptive_test_result[
                "area_hotspot_coverage"
            ]
            / fixed_test_result[
                "area_hotspot_coverage"
            ]
        ),
        "comparison":
            "Percentage retained versus fixed capacity"
    },
    {
        "section": "Confidence audit",
        "metric": "Maximum absolute calibration bias",
        "system_value":
            maximum_absolute_confidence_bias,
        "comparator_value": 0.0,
        "relative_change_percent": np.nan,
        "comparison":
            "Across test confidence bands"
    },
    {
        "section": "Scenario analysis",
        "metric": "Mean local gain per test hour at 60%",
        "system_value": (
            test_scenario_gain_60
            / test_hours
        ),
        "comparator_value": np.nan,
        "relative_change_percent": np.nan,
        "comparison":
            "Non-causal hypothetical suppression scenario"
    }
])

display(headline_metrics)

,section,metric,system_value,comparator_value,relative_change_percent,comparison
0,Frozen forecast,Pressure capture at K=25,0.3993,0.2928,36.3661,Recent-24-hour baseline
1,Frozen forecast,Hotspot recall at K=25 and threshold=8,0.5024,0.3730,34.6895,Recent-24-hour baseline
2,Frozen forecast,NDCG at K=25,0.3834,0.2194,74.7707,Recent-24-hour baseline
3,Spatial holdout,Pressure capture at proportional K,0.3488,0.2575,35.4506,Unseen-zone recent-24-hour baseline
4,Spatial holdout,Hotspot recall at proportional K,0.4457,0.3267,36.4303,Unseen-zone recent-24-hour baseline
5,Temporal ablation,Pressure capture versus static heatmap,0.3868,0.3497,10.6353,Static heatmap-like model
6,Temporal ablation,Pressure capture versus no recurrence,0.3868,0.3671,5.3696,No-temporal-recurrence model
7,Adaptive dispatch,Mean patrols deployed at capacity 25,13.8254,25.0000,-44.6985,Forced fixed-capacity deployment
8,Adaptive dispatch,Direct pressure capture retained,0.3685,0.3878,95.0369,Percentage retained versus fixed capacity
9,Adaptive dispatch,Area hotspot coverage retained,0.6797,0.6941,97.9287,Percentage retained versus fixed capacity


In [19]:
# creates a self-contained package for the prototype and final report.

PACKAGE_DIR = OUTPUT_DIR / "prototype_package"
PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

prototype_parquet_path = (
    PACKAGE_DIR
    / "prototype_dispatch_recommendations.parquet"
)

prototype_csv_path = (
    PACKAGE_DIR
    / "prototype_dispatch_recommendations.csv"
)

latest_csv_path = (
    PACKAGE_DIR
    / "latest_dispatch_recommendations.csv"
)

schedule_csv_path = (
    PACKAGE_DIR
    / "prototype_hourly_dispatch_schedule.csv"
)

geojson_path = (
    PACKAGE_DIR
    / "prototype_dispatch_recommendations.geojson"
)

schema_path = (
    PACKAGE_DIR
    / "prototype_schema_manifest.csv"
)

metrics_path = (
    PACKAGE_DIR
    / "consolidated_evaluation_summary.csv"
)

model_card_path = (
    PACKAGE_DIR
    / "MODEL_CARD.md"
)


prototype_dispatch.to_parquet(
    prototype_parquet_path,
    index=False
)

prototype_dispatch.to_csv(
    prototype_csv_path,
    index=False
)

latest_prototype_dispatch.to_csv(
    latest_csv_path,
    index=False
)

adaptive_schedule.to_csv(
    schedule_csv_path,
    index=False
)

headline_metrics.to_csv(
    metrics_path,
    index=False
)


field_descriptions = {
    "recommendation_id":
        "Unique target-hour and zone recommendation identifier.",
    "target_time":
        "Start of the forecasted one-hour interval in IST.",
    "selection_order":
        "Balanced patrol priority within the hour.",
    "dispatch_tier":
        "Highest, secondary or extended coverage priority.",
    "recommended_patrols":
        "Total recommendations issued for the target hour.",
    "zone_index":
        "Internal 250 m zone identifier.",
    "zone_250m":
        "Human-readable 250 m grid identifier.",
    "latitude":
        "Zone-centroid latitude.",
    "longitude":
        "Zone-centroid longitude.",
    "spatial_priority_score":
        "Forecast score with validated distance context.",
    "calibrated_pressure":
        "Expected next-hour local incident pressure.",
    "hotspot_probability":
        "Calibrated probability of at least eight incidents.",
    "observation_confidence":
        "Reliability of the historical observation process.",
    "confidence_tier":
        "Low, medium or high observation confidence.",
    "verification_required":
        "Whether officers should verify conditions before action.",
    "patrol_action":
        "Recommended operational response.",
    "reason_codes":
        "Pipe-separated machine-readable explanation codes.",
    "evidence_summary":
        "Human-readable evidence explanation.",
    "local_pressure_removed_60pct":
        "Hypothetical local pressure removed under 60% suppression.",
    "scenario_scope":
        "Warning that scenario outputs are non-causal.",
    "spatial_context_role":
        "Warning that neighbourhood context is not propagation."
}

schema_manifest = pd.DataFrame({
    "field": prototype_dispatch.columns,
    "dtype": [
        str(prototype_dispatch[col].dtype)
        for col in prototype_dispatch.columns
    ],
    "nullable": [
        bool(
            prototype_dispatch[col]
            .isna()
            .any()
        )
        for col in prototype_dispatch.columns
    ],
    "description": [
        field_descriptions.get(
            col,
            "Supporting forecast, evidence or operational field."
        )
        for col in prototype_dispatch.columns
    ]
})

schema_manifest.to_csv(
    schema_path,
    index=False
)


geojson_features = []

geojson_property_columns = [
    "recommendation_id",
    "target_time",
    "selection_order",
    "dispatch_tier",
    "zone_index",
    "spatial_priority_score",
    "calibrated_pressure",
    "hotspot_probability",
    "observation_confidence",
    "confidence_tier",
    "verification_required",
    "patrol_action",
    "reason_codes",
    "evidence_summary",
    "local_pressure_removed_60pct"
]

for row in prototype_dispatch.itertuples(
    index=False
):
    row_dict = row._asdict()

    properties = {}

    for col in geojson_property_columns:
        value = row_dict.get(col)

        if isinstance(value, pd.Timestamp):
            value = value.isoformat()
        elif isinstance(value, np.generic):
            value = value.item()

        properties[col] = value

    geojson_features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [
                float(row_dict["longitude"]),
                float(row_dict["latitude"])
            ]
        },
        "properties": properties
    })

with open(
    geojson_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "type": "FeatureCollection",
            "features": geojson_features
        },
        file,
        ensure_ascii=False
    )


model_card = f"""# PARK-GUARD Model Card

## Intended use

PARK-GUARD forecasts parking-incident pressure for 250 m zones one hour ahead and recommends up to 25 patrol locations per hour.

It is intended as a decision-support system for traffic-police and smart-city control rooms.

## Core configuration

- Spatial unit: 250 m grid
- Temporal unit: 1 hour
- Primary hotspot definition: at least 8 incidents
- Dispatch threshold: {MIN_DISPATCH_SCORE:.6f}
- Patrol balancing alpha: 0.60
- Patrol coverage radius: 500 m
- Low-confidence threshold: 0.755336
- High-confidence threshold: 0.907102

## Main test results

- Pressure capture at K=25: {official_result['pressure_capture_at_k_model']:.4f}
- Recent-24-hour baseline pressure capture: {official_result['pressure_capture_at_k_baseline']:.4f}
- Hotspot recall at K=25: {official_result['hotspot_recall_at_k_model']:.4f}
- Recent-24-hour baseline hotspot recall: {official_result['hotspot_recall_at_k_baseline']:.4f}
- NDCG at K=25: {official_result['mean_ndcg_at_k_model']:.4f}

## Geographic generalization

The spatial-holdout model achieved pressure capture of {spatial_model_result['pressure_capture_at_proportional_k']:.4f} and hotspot recall of {spatial_model_result['hotspot_recall_at_proportional_k']:.4f} on zones excluded from training.

## Adaptive deployment

At nominal capacity 25, adaptive dispatch deployed an average of {adaptive_test_result['mean_patrols_deployed']:.2f} patrols per hour.

It retained {100 * adaptive_test_result['direct_pressure_capture'] / fixed_test_result['direct_pressure_capture']:.2f}% of direct pressure capture and {100 * adaptive_test_result['area_hotspot_coverage'] / fixed_test_result['area_hotspot_coverage']:.2f}% of area hotspot coverage.

## Limitations and safeguards

1. Incident reports are a proxy for parking pressure, not direct traffic-speed or congestion measurements.
2. Enforcement-benefit fields are hypothetical suppression scenarios, not causal treatment-effect estimates.
3. Neighbourhood exposure is used for ranking and patrol coverage; it must not be described as learned congestion propagation.
4. The learned directional influence graph was rejected after failing holdout validation.
5. Low-confidence and cold-start recommendations require on-site verification.
6. The system must not automatically issue penalties or infer legal guilt.
7. Reporting intensity varies geographically and may reflect enforcement practices as well as underlying parking behaviour.
"""

with open(
    model_card_path,
    "w",
    encoding="utf-8"
) as file:
    file.write(model_card)


package_files = [
    prototype_parquet_path,
    prototype_csv_path,
    latest_csv_path,
    schedule_csv_path,
    geojson_path,
    schema_path,
    metrics_path,
    model_card_path
]

package_manifest_rows = []

for path in package_files:
    package_manifest_rows.append({
        "filename": path.name,
        "size_mb": (
            path.stat().st_size
            / 1024**2
        )
    })

package_manifest = pd.DataFrame(
    package_manifest_rows
)

manifest_path = (
    PACKAGE_DIR
    / "package_manifest.csv"
)

package_manifest.to_csv(
    manifest_path,
    index=False
)

package_files.append(manifest_path)

zip_path = (
    OUTPUT_DIR
    / "park_guard_prototype_package.zip"
)

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in package_files:
        archive.write(
            path,
            arcname=path.name
        )

display(package_manifest)

print("Package created:", zip_path)
print(
    "Package size MB:",
    zip_path.stat().st_size / 1024**2
)

,filename,size_mb
0,prototype_dispatch_recommendations.parquet,0.6537
1,prototype_dispatch_recommendations.csv,9.9760
2,latest_dispatch_recommendations.csv,0.0027
3,prototype_hourly_dispatch_schedule.csv,0.0727
4,prototype_dispatch_recommendations.geojson,11.6079
5,prototype_schema_manifest.csv,0.0033
6,consolidated_evaluation_summary.csv,0.0016
7,MODEL_CARD.md,0.0018


Package created: /kaggle/working/parking_phase7/park_guard_prototype_package.zip
Package size MB: 2.1186866760253906


In [20]:
latest_schedule_row = (
    adaptive_schedule
    .sort_values("target_time")
    .iloc[-1]
)

print("Latest target time:", latest_schedule_row["target_time"])
print(
    "Latest patrol count:",
    int(latest_schedule_row["recommended_patrols"])
)
print(
    "Deployment status:",
    latest_schedule_row["deployment_status"]
)

Latest target time: 2024-04-08 23:00:00+05:30
Latest patrol count: 3
Deployment status: deployment_recommended
